# Uygulama: Yeni Cümle için Dikkat Kümesi Tahmini ve Benzer Cümle Önerisi
Bu projede, bir cümledeki **verb → token** attention vektörü çıkarılır. Ardından, daha önce K-Means ile kümelediğimiz dikkat yapıları arasında bu yeni cümle nereye düşer, hangi kümeye aittir, ve bu kümeye en yakın örnek cümleler nelerdir, belirlenir.

Bu işlem sayesinde attention matrislerinden **yapısal benzerlik analizi yapan bir uygulama** gerçekleştirilmiş olur.

In [1]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from torch.nn.functional import pad

In [10]:
import pandas as pd
import torch
from torch.nn.functional import pad

df = pd.read_excel('/content/dataset.xlsx')
df['tokens'] = df['tokens'].apply(eval)
df['labels'] = df['labels'].apply(eval)

attention_vectors = []
sentences = []

for _, row in df.iterrows():
    sentence = row['sentence']
    labels = row['labels']
    try:
        verb_index = labels.index('verb') + 1  # tokenizer inputunda [CLS] varsa onu hesaba kat
        vec, tokens = get_attention_vector(sentence, verb_index)
        attention_vectors.append(vec)
        sentences.append(sentence)
    except:
        continue

max_len = max(len(v) for v in attention_vectors)
def pad_vector(v, length):
    v = torch.tensor(v)
    return pad(v, (0, length - len(v))).numpy()

X = np.stack([pad_vector(v, max_len) for v in attention_vectors])
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)

XLMRobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


In [2]:
model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)
model.eval()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine

In [8]:
def get_attention_vector(sentence, verb_index):
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    attentions = outputs.attentions
    avg_attention = torch.stack(attentions).mean(dim=0)[0].mean(dim=0)
    return avg_attention[verb_index].cpu().numpy(), tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

In [9]:
# Uygulama fonksiyonu
def classify_and_suggest(sentence, verb_index):
    vec, tokens = get_attention_vector(sentence, verb_index)
    vec_pad = pad_vector(vec, max_len).reshape(1, -1)
    cluster_id = kmeans.predict(vec_pad)[0]
    print(f'Bu cümle {cluster_id}. kümeye ait.')

    sims = cosine_similarity(vec_pad, X[kmeans.labels_ == cluster_id])[0]
    top_idx = sims.argsort()[-3:][::-1]  # en benzer 3 cümle

    print('\nBenzer dikkat yapısına sahip cümleler:')
    cluster_sentences = [s for i, s in enumerate(sentences) if kmeans.labels_[i] == cluster_id]
    for i in top_idx:
        print(f'- {cluster_sentences[i]}')

In [11]:
# Test örneği
test_sentence = 'Ali topu attı.'
classify_and_suggest(test_sentence, verb_index=3)  # 'attı' 3. token

Bu cümle 2. kümeye ait.

Benzer dikkat yapısına sahip cümleler:
- Usta perdeyi taktı.
- Hoca notları girdi.
- Polis genci tutukladı.


In [ ]:
# kullanıcıdan cümle al
print('\nYeni Cümle Sınıflandırma')
user_input = input('Bir Türkçe cümle girin (örn: Ayşe kalemi aldı.): ')
tokens = tokenizer.tokenize(user_input)
print('\nTokenlar:', tokens)
print('Fiilin token indeksini elle girmeniz gerekiyor. Token dizinleri:')
for i, tok in enumerate(tokens):
    print(f'{i}: {tok}')
verb_idx = int(input('\nFiilin indexini girin: '))

classify_and_suggest(user_input, verb_idx)


--- Yeni Cümle Sınıflandırma ---
Bir Türkçe cümle girin (örn: Ayşe kalemi aldı.): Zeytin mamasını bitirdi.

Tokenlar: ['▁Zeytin', '▁ma', 'masını', '▁bitir', 'di', '.']
Fiilin token indeksini elle girmeniz gerekiyor. Token dizinleri:
0: ▁Zeytin
1: ▁ma
2: masını
3: ▁bitir
4: di
5: .

Fiilin indexini girin: 3
Bu cümle 2. kümeye ait.

Benzer dikkat yapısına sahip cümleler:
- Ağaçlar çiçek açtı.
- İnci çayı döktü.
- Öğrenci otobüsü kaçırdı.


In [18]:
def print_example_sentences_by_cluster(sentences, labels, n=3):
    """
    Her kümeye ait en fazla n adet örnek cümleyi yazdırır.
    :param sentences: Cümle listesi
    :param labels: KMeans çıkışı olan cluster ID'leri
    :param n: Her kümeden gösterilecek maksimum örnek sayısı
    """
    clusters = sorted(set(labels))
    for cluster_id in clusters:
        print(f"\n🔹 Küme {cluster_id} örnek cümleleri:")
        count = 0
        for i, sentence in enumerate(sentences):
            if labels[i] == cluster_id:
                print(f"- {sentence}")
                count += 1
                if count == n:
                    break


In [20]:
print_example_sentences_by_cluster(sentences, kmeans.labels_, n=5)



🔹 Küme 0 örnek cümleleri:
- Kedi sandalyeyi devirdi.
- Tarık kapıya koştu.
- Ufuk çöpü attı.
- Kedi resmi yırttı.
- Martı pencereyi görmemiş.

🔹 Küme 1 örnek cümleleri:
- Ali uçurtma uçurdu.
- Babası kapıyı tıkladı.
- Zeynep topu buldu.
- İclal yolu yürüdü.
- İclal sunum yaptı.

🔹 Küme 2 örnek cümleleri:
- Ali defteri aldı.
- Ayşe elmayı yedi.
- Mehmet ağacı gördü.
- Çocuk deneme yazdı.
- Anne fare peyniri yedi.
